# Rotated surface code — **subset** joint lattice-surgery measurement (obstacle-aware routing)

Measure the joint `M(∏ᵢ P̄ᵢ)` of only a **subset** of many placed patches; the non-target patches are
**obstacles** the routed `d`-wide ancilla bus avoids.  Routing / construction logic lives in
[`lightstim/qec_code/surface_code/rotated/subset_routing.py`](../../lightstim/qec_code/surface_code/rotated/subset_routing.py).

**Input** — just the native `PatchSpec(name, origin, distance, measured_logical, orientation)` **array**
(the same class the N-patch API uses) + a `target = [(name, "X"|"Z"), …]` list saying which patches are
measured (the rest are **obstacles**).  `origin_of(a,b,d)` places a patch on coarse cell `(a,b)`.
**No hand-written corridor:** `route_and_build(patches, target)` finds *and* verifies the bus
automatically — propose-and-verify with retry, trying the **standard** construction first and the
**convex-corner cut** as a fallback (see the Summary).

**Every example PASSES** the strict **12-point acceptance** gate; bent geometries fall back automatically to
the **convex-corner cut** (removing the bend's outer qubit so the convex 90° corner becomes a genuine
**weight-3** stabilizer — fully CSS, no twist).  Each example shows **four** things: **(1)** the auto-routed
path, **(2)** the 12-point acceptance checklist, **(3)** the data-qubit layout, **(4)** the Stim
`detslice-with-ops` diagram.

In [ ]:
import sys, os, itertools
sys.path.insert(0, os.path.abspath('../..'))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon, Circle, Rectangle
from IPython.display import SVG, display

# PatchSpec is lightstim's NATIVE spec class (defined in .bent_layout, exported at the rotated package
# level) -- the very same input the N-patch generator and the N-patch notebook use.  NOT a wrapper.
from lightstim.qec_code.surface_code.rotated import PatchSpec
from lightstim.qec_code.surface_code.rotated.subset_routing import (
    route_and_build, collision_report, cell, _specs_to_cells)
from lightstim.qec_code.surface_code.rotated.multi_patch import _int_symplectic, _icommute, _IntBasis

D = 3   # code distance for the demos


def origin_of(a, b, d):
    '''Data-qubit origin (bus-facing (1,1)-corner) of coarse cell (a, b) at pitch 2d -- i.e. the
    PatchSpec.origin that places a distance-d patch on that cell.  Defined locally here so the notebook
    writes patch placements on the coarse routing grid without importing an extra helper.'''
    return (1 + 2 * d * a, 1 + 2 * d * b)

In [ ]:
def acceptance_of(lay):
    """The strict 12-point acceptance dict computed directly from a MultiPatchLayout (standard OR cut)."""
    v = lay.verify()
    isv, nn = _int_symplectic(lay.data)
    S = [isv(c['pauli']) for c in lay.checks]
    B = _IntBasis()
    for sv_ in S:
        B.add(sv_)
    logvecs = [isv({c: P for c in sup}) for _, P, sup in lay.logicals]
    log_ok = all(_icommute(Lv, sv_, nn) for Lv in logvecs for sv_ in S)
    joint = 0
    for Lv in logvecs:
        joint ^= Lv
    chain = lay.readout_chain
    prod = 0
    for c in lay.checks:
        if c['syn'] in chain:
            prod ^= isv(c['pauli'])
    chain_ok = bool(chain) and prod == joint
    N = len(lay.logicals)
    items = {
        'target logicals commute with all stabilizers': log_ok,
        'remaining logical dof == N-1': v['logical_count'],
        'full joint in span': v['joint'],
        'no single logical measured': v['no_single'],
        'no proper sub-product measured': v['no_subjoint'],
        'no weight-1 leftover logical': v['no_weight1_logical'],
        'all stabilizers commute': v['commute'],
        'no Y / no twist': v['no_twist'],
        'no MPP': v['no_mpp'],
        'DEM valid': v['dem_valid'],
        'no tick collision': v['no_tick_collision'],
        'readout chain exists and product == joint': chain_ok,
    }
    return dict(accept=all(bool(x) for x in items.values()), items=items, N=N, data=len(lay.data),
                n_stab=len(lay.checks), k=len(lay.data) - B.rank, readout_chain_len=len(chain))


def print_acceptance(a, label):
    """(2) The 12-point PASS checklist."""
    n_ok = sum(1 for x in a['items'].values() if x is True)
    print(f"{label}   ->  {'PASS' if a['accept'] else 'FAIL'}   ({n_ok}/12 conditions)")
    print(f"    data qubits = {a['data']}   independent stabilizers (rank) = {a['n_stab']}   "
          f"readout chain = {a['readout_chain_len']} stabs")
    print(f"    remaining logical dof  =  data - stabilizers  =  {a['data']} - {a['n_stab']}  =  "
          f"{a['k']}   (want N-1 = {a['N']-1})")
    for k, x in a['items'].items():
        print(f"    [{'True ' if x is True else str(x)}] {k}")

In [ ]:
def draw_path(patches, target, route, title, d=D):
    """(1) The routed path on the coarse-cell grid: target patches coloured (X̄ blue / Z̄ orange),
    obstacles grey-hatched, the ancilla corridor in green."""
    patch_at, _o, _d = _specs_to_cells(patches, target)
    tset = dict(target)
    fig, ax = plt.subplots(figsize=(8.6, 4.6))
    def box(a, b, **kw):
        qs = cell(a, b, d); x0, x1 = min(q[0] for q in qs), max(q[0] for q in qs)
        y0, y1 = min(q[1] for q in qs), max(q[1] for q in qs)
        ax.add_patch(Rectangle((x0 - 1, y0 - 1), x1 - x0 + 2, y1 - y0 + 2, **kw))
        return (x0 + x1) / 2, (y0 + y1) / 2
    for (a, b) in route:                                        # ancilla corridor
        box(a, b, facecolor='#bfe6bf', edgecolor='#2ca02c', lw=1.4, zorder=1)
    for p in patches:                                           # patches
        a, b = patch_at[p.name]
        if p.name in tset:
            cx, cy = box(a, b, facecolor=('#6f9cf0' if tset[p.name] == 'X' else '#ef7a45'),
                         edgecolor='white', lw=1, zorder=2)
            ax.text(cx, cy, p.name, ha='center', va='center', color='white', fontsize=9, fontweight='bold', zorder=3)
        else:
            cx, cy = box(a, b, facecolor='0.86', edgecolor='0.6', hatch='xxx', lw=0.5, zorder=1)
            ax.text(cx, cy, p.name, ha='center', va='center', color='0.4', fontsize=8, zorder=3)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.autoscale_view(); ax.margins(0.08); ax.axis('off')
    ax.set_title(f"{title}  —  routed path (coarse cells)", fontsize=11)
    ax.legend(handles=[mpatches.Patch(color='#6f9cf0', label='target X̄'),
                       mpatches.Patch(color='#ef7a45', label='target Z̄'),
                       mpatches.Patch(facecolor='#bfe6bf', edgecolor='#2ca02c', label='ancilla corridor'),
                       mpatches.Patch(facecolor='0.86', edgecolor='0.6', hatch='xxx', label='obstacle')],
              loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8.5, frameon=False)
    plt.tight_layout(); plt.show()


def show(lay, title):
    """(3) Data-qubit layout: X/Z/mixed stabilizers, data qubits, logicals, gold readout chain."""
    COL = {'X': '#e23b3b', 'Z': '#2f6fd0', 'M': '#8b3fd0'}
    FILL = {'X': '#f6b8b8', 'Z': '#b8cdf0', 'M': '#d9c2f2'}; GOLD = '#f0a000'
    CHAIN = lay.readout_chain
    allc = [c for ch in lay.checks for c in ch['corners']] + list(lay.data)
    x0, x1 = min(p[0] for p in allc) - 1.6, max(p[0] for p in allc) + 1.6
    y0, y1 = min(p[1] for p in allc) - 1.6, max(p[1] for p in allc) + 1.6
    fig, ax = plt.subplots(figsize=(min(13, 0.85 + (x1 - x0) * 0.42), min(13, 0.85 + (y1 - y0) * 0.42)))
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor='#f6f6f9', ec='none', zorder=0))
    for ch in lay.checks:
        t, pts, syn = ch['type'], ch['corners'], ch['syn']; hl = syn in CHAIN; ec = GOLD if hl else COL[t]
        if len(pts) >= 3:
            cx, cy = np.mean([p[0] for p in pts]), np.mean([p[1] for p in pts])
            order = sorted(pts, key=lambda p: np.arctan2(p[1] - cy, p[0] - cx))
            ax.add_patch(Polygon(order, closed=True, facecolor=FILL[t], edgecolor=ec,
                                 lw=2.4 if hl else 0.6, alpha=0.9 if hl else 0.30, zorder=3 if hl else 2))
        elif len(pts) == 2:
            (a, b), (c, dd) = pts
            ax.plot([a, syn[0], c], [b, syn[1], dd], color=ec, lw=7 if hl else 5,
                    alpha=0.7 if hl else 0.28, solid_capstyle='round', zorder=3 if hl else 2)
        if t == 'M':
            for c, P in ch['pauli'].items():
                ax.plot([syn[0], c[0]], [syn[1], c[1]], color=COL[P], lw=2.4, zorder=5, alpha=0.9, solid_capstyle='round')
        ax.add_patch(Rectangle((syn[0] - 0.18, syn[1] - 0.18), 0.36, 0.36, facecolor=COL[t],
                     edgecolor=GOLD if hl else 'white', lw=2 if hl else 0.8, zorder=6, alpha=1 if hl else 0.55))
    for q in lay.data:
        ax.add_patch(Circle(q, 0.14, facecolor='#1a1a1a', edgecolor='white', lw=0.6, zorder=8))
    xshades = ['#c01616', '#7a0d0d', '#e0552a', '#9c1b5a', '#5a1b9c']; xi = 0
    for nm, P, sup in lay.logicals:
        col = '#13346e' if P == 'Z' else xshades[xi % len(xshades)]; xi += (P == 'X')
        srt = sorted(sup)
        ax.plot([q[0] for q in srt], [q[1] for q in srt], color=col, lw=5, zorder=10, solid_capstyle='round')
        mx, my = srt[len(srt) // 2]
        ax.text(mx, my - 0.8, fr'$\bar {P}_{{{nm[1:]}}}$', color=col, fontsize=13, fontweight='bold', ha='center', zorder=11,
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec=col, lw=1))
    handles = [mpatches.Patch(color=COL['X'], label='X stabilizer'), mpatches.Patch(color=COL['Z'], label='Z stabilizer'),
               mpatches.Patch(color=COL['M'], label='MIXED (XZ) wall'),
               mpatches.Patch(facecolor='#fff3d6', edgecolor=GOLD, lw=2, label='readout chain (product = joint)')]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.005), ncol=2, fontsize=9)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    ax.set_title(title, fontsize=12, pad=46); plt.tight_layout(); plt.show()


def run_example(patches, target, title):
    """Show the FOUR things for a subset joint: path, acceptance, layout, detslice SVG.
    ``patches`` is a PatchSpec array (targets + obstacles); ``target`` names the measured subset.
    The corridor is found **fully automatically** by ``route_and_build`` (propose-and-verify with
    retry: it enumerates obstacle-aware candidates and verifies each — standard first, then the
    convex-corner cut — returning the first valid one).  No hand-written ``route``."""
    tnames = {nm for nm, _ in target}
    r = route_and_build(patches, target)                            # AUTO route + build + verify
    if not r.ok:
        raise RuntimeError(f"route_and_build failed: {r.status} — {r.message}")
    route, lay = sorted(r.tree), r.layout
    how = 'standard construction' if r.how == 'standard' else f'convex-corner cut at {r.cut}'
    draw_path(patches, target, route, title)                        # (1) path (auto-routed)
    print_acceptance(acceptance_of(lay), f"{title}   [{how}]")       # (2) acceptance
    if any(p.name not in tnames for p in patches):
        print("    collision-clean (routed code vs obstacles):", collision_report(patches, target, route)['clean'])
    show(lay, title)                                                # (3) layout
    display(SVG(str(lay.build_circuit(rounds=2, p=0.0).diagram('detslice-with-ops-svg'))))   # (4) detslice

## Example 1 — subset joint `M(X̄₁ Z̄₂ Z̄₃)`: 9 patches, measure only 3

Nine `PatchSpec`s; `target` selects `X1` + `Z2` + `Z3`; the six `B*` are **obstacles** the bus threads
between (`keepout = 1`).  `run_example` finds and verifies the corridor automatically — no `route` argument.

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(2,  1, D), D, "Z", "X_horizontal"),
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B1", origin_of(0,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B3", origin_of(4,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]
target  = [("X1", "X"), ("Z2", "Z"), ("Z3", "Z")]
run_example(patches, target, "Subset joint  M(X̄1 Z̄2 Z̄3)")       # corridor found automatically

## Example 2 — simplest subset joint `M(X̄₁ Z̄₂)` with obstacles

`X1` and `Z2` are the targets; `B1`, `B2` are **obstacle** patches on the chip.  Given a free straight
corridor between the two targets, `route_and_build` prefers it (it tries candidates shortest-first and
takes the first that verifies), so this auto-routes to a clean **straight** trunk with a mixed wall at
`Z2` — the corner cut isn't needed here (it appears in Example 4, where the geometry forces a bend).

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(3,  0, D), D, "Z", "X_horizontal"),
           PatchSpec("B1", origin_of(0, -2, D), D, "X", "X_horizontal"),
           PatchSpec("B2", origin_of(3, -2, D), D, "X", "X_horizontal")]
target  = [("X1", "X"), ("Z2", "Z")]                      # B1, B2 are obstacles
run_example(patches, target, "Subset joint  M(X̄1 Z̄2)")   # corridor found automatically

## Example 3 — subset joint `M(X̄₁ Z̄₃)`: drop `Z2` (obstacle between targets)

Same nine patches, now `target` selects only `X1` and `Z3`; `Z2` becomes an obstacle **between** them.
`route_and_build` auto-finds a clean below-attach corridor (`X1` exits its bottom face and runs straight
across to `Z3`) and verifies it with the **standard** construction — a genuine *non-adjacent* subset.

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(2,  1, D), D, "Z", "X_horizontal"),
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B1", origin_of(0,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B3", origin_of(4,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]
target  = [("X1", "X"), ("Z3", "Z")]                      # drop Z2 -> it is now an obstacle
run_example(patches, target, "Subset joint  M(X̄1 Z̄3)  (drop Z2)")   # corridor found automatically

## Example 4 — add `B2` as a third target: `M(X̄₁ X̄_B2 Z̄₃)`

Building on Example 3's chip, with two small changes so `B2` becomes measurable: **move `Z2` one cell
right** `(2,1) → (3,1)`, and **remove `B1` and `B3`** (they flanked `B2`).  Now `B2(2,3)` is a **third
measured target** contributing `X̄`, giving `M(X̄₁ X̄_B2 Z̄₃)` (`N = 3`, so `dof = N−1 = 2`): `X1` and `B2`
fuse on the X-bus, `Z3` attaches through a mixed wall, and `Z2`/`B4`/`B5`/`B6` are obstacles.  Reaching
`B2` up high **forces a bend**, so no standard candidate verifies — `route_and_build` falls through to the
**convex-corner cut** automatically (removes the bend's outer qubit → weight-3 corner, fully CSS, no twist).

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(3,  1, D), D, "Z", "X_horizontal"),   # moved right (was (2,1)); obstacle
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),   # now a target (was an obstacle)
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),   # obstacle
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),   # obstacle
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]   # obstacle
target  = [("X1", "X"), ("B2", "X"), ("Z3", "Z")]          # measure M(X̄1 X̄_B2 Z̄3)
run_example(patches, target, "Subset joint  M(X̄1 X̄_B2 Z̄3)")   # bent corridor + corner-cut, all automatic

## Summary

* **Input** is just a `PatchSpec` array + a `target` list (which patches are measured; the rest are
  obstacles) — the same native form as the N-patch API.  **No hand-written corridor:** `route_and_build`
  finds and verifies it automatically.
* **`route_and_build` = propose-and-verify with retry** (the algorithm this notebook is built on):
  enumerate obstacle-aware candidate corridors (any anchor face, shortest-first) and verify each against
  the GF(2) oracle — **standard construction first, then the convex-corner cut** as a fallback — returning
  the first valid one.  So a clean straight/L route always wins when one exists (Examples 1–3), and the
  corner cut is used only when the geometry genuinely forces a bend (Example 4).
* **Every example passes** the strict 12-point acceptance oracle and is collision-clean where obstacles are
  present.  Each shows four things: the auto-routed path, the acceptance checklist, the data-qubit layout,
  and the Stim detslice diagram.
* **The convex-corner cut** (remove the bend's outer bus qubit so the convex 90° corner becomes a weight-3
  stabilizer — fully CSS, no twist) is what lets a bent joint verify; `route_and_build` applies it
  automatically whenever no standard candidate leaves `dof == N−1`.